# 🎯 03 — Customer Segmentation
## Bank Marketing Campaign

**Objective:** Identify distinct customer segments using clustering to understand which groups respond best to marketing campaigns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import sys
sys.path.append('..')
from src.utils import load_data

sns.set_style('whitegrid')
%matplotlib inline

In [ ]:
df = load_data('../data/bank-additional-full.csv')
df['subscribed'] = (df['y'] == 'yes').astype(int)

## 1. Data Preparation for Clustering

In [ ]:
# Select features for clustering
cluster_features = ['age', 'balance', 'duration', 'campaign', 'pdays', 'previous']
df_cluster = df[cluster_features].dropna()

# Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster)
print(f'Clustering dataset shape: {X_scaled.shape}')

## 2. Optimal Number of Clusters (Elbow + Silhouette)

In [ ]:
inertias = []
silhouettes = []
K_range = range(2, 9)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    inertias.append(kmeans.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method')

axes[1].plot(K_range, silhouettes, 'ro-')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Analysis')

plt.tight_layout()
plt.savefig('../figures/clustering_optimal_k.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Final Clustering

In [ ]:
# Fit with optimal k (adjust based on elbow/silhouette)
k_optimal = 4  # adjust after seeing the plots
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
df_cluster['cluster'] = kmeans.fit_predict(X_scaled)
df_cluster['subscribed'] = df.loc[df_cluster.index, 'subscribed'].values

print(f'Cluster sizes:\n{df_cluster["cluster"].value_counts().sort_index()}')

## 4. Segment Profiling

In [ ]:
# Profile each cluster
profile = df_cluster.groupby('cluster').agg({
    'age': 'mean',
    'balance': 'mean',
    'duration': 'mean',
    'campaign': 'mean',
    'previous': 'mean',
    'subscribed': ['mean', 'count']
}).round(2)

profile.columns = ['Avg Age', 'Avg Balance', 'Avg Duration', 'Avg Campaigns', 'Avg Previous', 'Conversion Rate', 'Count']
profile

In [ ]:
# Visualize conversion by cluster
fig, ax = plt.subplots(figsize=(8, 5))
conv_by_cluster = df_cluster.groupby('cluster')['subscribed'].mean()
conv_by_cluster.plot(kind='bar', ax=ax, color=['#e74c3c', '#3498db', '#2ecc71', '#f39c12'][:k_optimal])
ax.set_ylabel('Conversion Rate')
ax.set_title('Conversion Rate by Customer Segment')
ax.axhline(y=df_cluster['subscribed'].mean(), color='black', linestyle='--', label='Overall avg')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/conversion_by_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

## Key Takeaways

1. ...
2. ...
3. ...